In [1]:
spark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder\
.appName("Read Modes in Spark")\
.getOrCreate()

26/08/26 17:30:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
!hadoop fs -cat /data/customers_with_errors.csv

customer_id,name,city,state,country,registration_date,is_active
1,John Doe,New York,NY,USA,2023-10-10,True
2,Jane,Los,Angeles,CA,USA,2023-08-1,False
3,Bob,Chicago,IL,USA,2023-07-20,True
4,Alice,,TX,USA,2023-07-20,True
5,Houston,TX,USA,2023-05-05,False
6,Charlie,San,Francisco,CA,USA,INVALIDE_DATE,True
7,David,Seattle,WA,USA,2023-04-22,INVALID


In [5]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,BooleanType,DateType

schema = StructType([
    StructField("customer_id",IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("city",StringType(),True),
    StructField("state",StringType(),True),
    StructField("country",StringType(),True),
    StructField("registration_date",DateType(),True),
    StructField("is_active",BooleanType(),True),
])

In [10]:
hdfs_file_path = '/data/customers_with_errors.csv'

df_failfast = spark.read\
.format('csv')\
.option('header',"true")\
.schema(schema)\
.option('mode','DROPMALFORMED')\
.load(hdfs_file_path)

df_failfast.show()

+-----------+--------+--------+-----+-------+-----------------+---------+
|customer_id|    name|    city|state|country|registration_date|is_active|
+-----------+--------+--------+-----+-------+-----------------+---------+
|          1|John Doe|New York|   NY|    USA|       2023-10-10|     true|
|          3|     Bob| Chicago|   IL|    USA|       2023-07-20|     true|
|          4|   Alice|    NULL|   TX|    USA|       2023-07-20|     true|
+-----------+--------+--------+-----+-------+-----------------+---------+



In [11]:
hdfs_file_path = '/data/customers_with_errors.csv'

df_failfast = spark.read\
.format('csv')\
.option('header',"true")\
.schema(schema)\
.option('mode','PERMISSED')\
.load(hdfs_file_path)

df_failfast.show()

26/08/26 17:51:09 WARN ParseMode: PERMISSED is not a valid parse mode. Using PERMISSIVE.
26/08/26 17:51:09 WARN ParseMode: PERMISSED is not a valid parse mode. Using PERMISSIVE.


+-----------+--------+--------+---------+----------+-----------------+---------+
|customer_id|    name|    city|    state|   country|registration_date|is_active|
+-----------+--------+--------+---------+----------+-----------------+---------+
|          1|John Doe|New York|       NY|       USA|       2023-10-10|     true|
|          2|    Jane|     Los|  Angeles|        CA|             NULL|     NULL|
|          3|     Bob| Chicago|       IL|       USA|       2023-07-20|     true|
|          4|   Alice|    NULL|       TX|       USA|       2023-07-20|     true|
|          5| Houston|      TX|      USA|2023-05-05|             NULL|     NULL|
|          6| Charlie|     San|Francisco|        CA|             NULL|     NULL|
|          7|   David| Seattle|       WA|       USA|       2023-04-22|     NULL|
+-----------+--------+--------+---------+----------+-----------------+---------+



In [12]:
spark.stop()